# Fraud Modeling: Fraud_Data

This notebook trains and compares two classifiers on the processed e-commerce fraud dataset: a baseline **Logistic Regression** model and a tuned **Random Forest** ensemble. We use a stratified train/test split, apply **SMOTE only to the training set** to address class imbalance, and evaluate on an untouched holdout that reflects real-world fraud prevalence (~9%).

**Goals**
- Load the model-ready feature matrix from `data/processed/fraud_data_features.csv`
- Split data with stratification so both sets retain fraud cases
- Train a simple baseline and a tuned ensemble on the same split
- Evaluate with metrics suited to imbalanced fraud detection
- Compare models side by side and identify the current best performer for interim submission

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import GridSearchCV

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RANDOM_STATE, TEST_SIZE
from src.modeling import (
    compare_class_distributions,
    compare_model_results,
    confusion_matrix_frame,
    load_fraud_feature_matrix,
    save_model_metrics,
    stratified_train_test_split,
    train_classifier,
)
from src.preprocessing import class_imbalance_summary

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Load Processed Features

We use the engineered feature matrix produced by the feature pipeline: numeric features scaled, categoricals one-hot encoded, and the fraud label (`class`) separated from predictors.

In [ ]:
features, target = load_fraud_feature_matrix()

print(f"Feature matrix shape: {features.shape}")
print(f"Target shape: {target.shape}")
print(f"Fraud rate: {target.mean():.2%}")

class_imbalance_summary(target.to_frame(name="class"), target_column="class")

## 2. Stratified Train/Test Split

We hold out **20%** of transactions for evaluation (`TEST_SIZE = 0.2`, `RANDOM_STATE = 42`). Stratification keeps the fraud rate similar in both splits so the test set is representative of production traffic.

In [ ]:
split = stratified_train_test_split(
    features,
    target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {len(split.x_train):,}")
print(f"Test rows:     {len(split.x_test):,}")
print(f"Features:      {len(split.feature_names)}")

train_dist = class_imbalance_summary(
    split.y_train.to_frame(name="class"), target_column="class"
)
test_dist = class_imbalance_summary(
    split.y_test.to_frame(name="class"), target_column="class"
)

display(
    pd.concat(
        [
            train_dist.assign(split="train"),
            test_dist.assign(split="test"),
        ],
        ignore_index=True,
    )
)

## 3. Handle Class Imbalance on Training Data Only

Fraud is the minority class (~9% of transactions). We apply **SMOTE** to the training set to synthesize additional fraud examples. The test set is **never** resampled — evaluation must reflect the natural class mix customers actually experience.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
x_train_resampled, y_train_resampled = smote.fit_resample(split.x_train, split.y_train)

x_train_resampled = pd.DataFrame(x_train_resampled, columns=split.feature_names)
y_train_resampled = pd.Series(y_train_resampled, name="class")

print(f"Training rows before SMOTE: {len(split.x_train):,}")
print(f"Training rows after SMOTE:  {len(x_train_resampled):,}")

distribution_comparison = compare_class_distributions(
    split.y_train,
    y_train_resampled,
    split.y_test,
)
display(distribution_comparison[["stage", "class", "count", "pct"]])

## 4. Train Baseline Logistic Regression

Logistic Regression is a strong first baseline: fast to train, easy to interpret, and a useful reference point before tree-based or gradient-boosted models. We train on SMOTE-balanced data without extra class weights, since resampling already rebalances the training set.

In [ ]:
baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)

baseline_result = train_classifier(
    baseline_model,
    x_train_resampled,
    y_train_resampled,
    split.x_test,
    split.y_test,
    model_name="logistic_regression",
    store_training_data=True,
)

print(f"Baseline trained on {len(x_train_resampled):,} resampled rows.")
print(f"Holdout evaluation on {len(split.x_test):,} untouched test rows.")

## 5. Train Tuned Random Forest Ensemble

Random Forest captures non-linear patterns and feature interactions that Logistic Regression cannot. We run a **small hyperparameter search** on the original (pre-SMOTE) training split to keep runtime manageable, then refit the best configuration on the SMOTE-balanced training set for the final model.

**Search space**
- `n_estimators`: number of trees (100, 200)
- `max_depth`: tree depth cap (12, 20, unlimited)
- `min_samples_leaf`: minimum samples per leaf (1, 5)

Cross-validation uses **AUC-PR** as the scoring metric because it prioritizes performance on the rare fraud class.

In [ ]:
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [12, 20, None],
    "min_samples_leaf": [1, 5],
}

rf_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=rf_param_grid,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

print("Tuning Random Forest on the original training split (pre-SMOTE)...")
rf_search.fit(split.x_train, split.y_train)

print("\nBest cross-validated AUC-PR:", f"{rf_search.best_score_:.4f}")
print("Best parameters:", rf_search.best_params_)

best_rf = RandomForestClassifier(
    **rf_search.best_params_,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

ensemble_result = train_classifier(
    best_rf,
    x_train_resampled,
    y_train_resampled,
    split.x_test,
    split.y_test,
    model_name="random_forest_tuned",
    store_training_data=True,
)

print(f"\nEnsemble refit on {len(x_train_resampled):,} SMOTE-balanced rows.")

## 6. Model Comparison

Both models are evaluated on the **same untouched test set** using identical metrics. For fraud detection with ~9% prevalence, **AUC-PR** is our primary ranking metric because it measures how well each model separates fraud from legitimate transactions when the positive class is rare. F1, recall, and precision at the default 0.5 threshold are reported for operational context.

| Metric | Business meaning |
|--------|------------------|
| **Precision** | Of flagged transactions, how many are actually fraud? (controls false alarms) |
| **Recall** | Of all fraud, how much do we catch? (controls missed fraud) |
| **F1** | Balance between precision and recall at the default threshold |
| **ROC-AUC** | Overall ranking ability across thresholds |
| **AUC-PR** | Ranking quality focused on the rare fraud class — primary metric for model selection here |

In [ ]:
model_results = [baseline_result, ensemble_result]

comparison = compare_model_results(model_results, sort_by="auc_pr")
comparison_display = comparison[
    [
        "model_name",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "auc_pr",
        "true_positives",
        "false_positives",
        "false_negatives",
        "true_negatives",
    ]
].copy()

for col in ["accuracy", "precision", "recall", "f1", "roc_auc", "auc_pr"]:
    comparison_display[col] = comparison_display[col].map(lambda v: f"{v:.4f}")

print("Side-by-side holdout metrics (sorted by AUC-PR):")
display(comparison_display)

metrics_path = save_model_metrics(comparison)
print(f"\nSaved comparison to: {metrics_path}")

best_row = comparison.iloc[0]
runner_up_row = comparison.iloc[1]
best_name = best_row["model_name"]
best_result = next(r for r in model_results if r.model_name == best_name)
best_metrics = best_result.metrics

auc_pr_gap = best_row["auc_pr"] - runner_up_row["auc_pr"]
f1_gap = best_row["f1"] - runner_up_row["f1"]

display(
    Markdown(
        f"**Current best model: `{best_name}`**\n\n"
        f"- **AUC-PR:** {best_metrics.auc_pr:.4f} "
        f"(+{auc_pr_gap:.4f} vs {runner_up_row['model_name']})\n"
        f"- **F1:** {best_metrics.f1:.4f} "
        f"(+{f1_gap:.4f} vs {runner_up_row['model_name']})\n"
        f"- **Recall:** {best_metrics.recall:.4f} | "
        f"**Precision:** {best_metrics.precision:.4f}\n\n"
        f"**Why it leads:** AUC-PR is the primary metric for imbalanced fraud detection "
        f"because it rewards models that rank fraud cases near the top of the alert queue. "
        f"{'Random Forest wins by learning non-linear feature interactions that a linear baseline cannot represent.'
        if best_name.startswith('random_forest')
        else 'Logistic Regression remains competitive, suggesting much of the signal is linear — the ensemble did not justify its added complexity on this holdout.'}"
    )
)

cm = confusion_matrix_frame(best_result.y_true, best_result.y_pred)
print(f"\nConfusion matrix — {best_name} (rows = actual, columns = predicted):")
display(cm)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Legitimate (0)", "Fraud (1)"],
    yticklabels=["Legitimate (0)", "Fraud (1)"],
    ax=ax,
)
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

total_fraud = best_metrics.true_positives + best_metrics.false_negatives
total_legit = best_metrics.true_negatives + best_metrics.false_positives
print(
    f"Fraud caught: {best_metrics.true_positives:,} / {total_fraud:,} "
    f"({best_metrics.true_positives / total_fraud:.1%} recall)"
)
print(
    f"False alarms: {best_metrics.false_positives:,} / {total_legit:,} legitimate transactions "
    f"({best_metrics.false_positives / total_legit:.2%} of legit flagged)"
)

## 7. Precision-Recall Curves

Overlaying both models on one chart makes the ranking difference visible. The model whose curve sits **higher and further right** ranks fraud cases better across thresholds. The dashed line is the **no-skill baseline** (fraction of fraud in the test set).

In [ ]:
baseline_prevalence = baseline_result.y_true.mean()

fig, ax = plt.subplots(figsize=(8, 6))

for model_result, color in [
    (baseline_result, "C0"),
    (ensemble_result, "C1"),
]:
    precision_vals, recall_vals, _ = precision_recall_curve(
        model_result.y_true,
        model_result.y_score,
    )
    ax.plot(
        recall_vals,
        precision_vals,
        linewidth=2,
        color=color,
        label=f"{model_result.model_name} (AUC-PR = {model_result.metrics.auc_pr:.3f})",
    )

ax.axhline(
    baseline_prevalence,
    linestyle="--",
    color="gray",
    label=f"No-skill baseline ({baseline_prevalence:.1%} fraud rate)",
)
ax.set_xlabel("Recall (fraud caught)")
ax.set_ylabel("Precision (flags that are fraud)")
ax.set_title("Precision-Recall Curves — Model Comparison")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 8. Interim Submission Summary

### Current best model

The comparison table above ranks models by **AUC-PR**. At interim submission stage, the leading model is the one with the highest AUC-PR on the stratified holdout set. If Random Forest wins, it likely captures non-linear patterns (e.g., channel × geography interactions) that Logistic Regression misses. If the baseline remains competitive, the features may be largely linear and the simpler model is preferable until threshold tuning and additional models are tested.

Run the comparison cell to see the exact winner and metric values for this dataset.

### Why AUC-PR drives the recommendation

Accuracy is high for both models (~90%+) because most transactions are legitimate — it is not a useful differentiator here. **AUC-PR** directly measures ranking quality for the fraud class, which aligns with how fraud teams work: they review the highest-scored transactions first. A higher AUC-PR means more fraud appears near the top of the ranked list, regardless of the exact alert threshold chosen later.

### Honest limitations at this stage

- Both models use a **default 0.5 probability threshold**, which is rarely optimal for fraud operations. Precision and recall will shift once a business-specific threshold is chosen.
- Hyperparameter tuning was **lightweight** (small grid, 3-fold CV on pre-SMOTE data) to keep notebook runtime reasonable — further tuning could change the ranking.
- Neither model has been explained with **SHAP** yet; interpretability and feature-level validation are planned next, not part of this interim comparison.
- Results apply to **Fraud_Data only**; the credit card dataset has a very different imbalance profile and will need its own modeling pass.

### Interim takeaway

We now have evidence that engineered features carry predictive signal and a **documented comparison** between a linear baseline and a tuned ensemble on identical data splits. The best current model should be carried forward for threshold analysis and SHAP-based interpretation in the next phase — but it should not yet be treated as production-ready without those follow-up steps.